# Regularization

## Data Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../../jupyter_notebooks/08-Linear-Regression-Models/Advertising.csv')

In [3]:
X = df.drop('sales',axis=1)

In [4]:
y = df['sales']

### Create polynomial features

In [5]:
from sklearn.preprocessing import PolynomialFeatures

In [6]:
polynomial_converter=PolynomialFeatures(degree=3,include_bias=False)

In [7]:
poly_features = polynomial_converter.fit_transform(X)

In [8]:
X.shape

(200, 3)

In [9]:
poly_features.shape

(200, 19)

### Make the usual test split 70% / 30%

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train,X_test,y_train,y_test=train_test_split(poly_features,y,test_size=0.3,random_state=101)

In [12]:
X_train.shape

(140, 19)

### Prepare rescaler with only Training data

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler = StandardScaler()

In [15]:
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


### Rescale both training and test sets

In [16]:
X_train = scaler.transform(X_train)

In [17]:
X_test = scaler.transform(X_test)

In [18]:
X_train[0]

array([ 0.49300171, -0.33994238,  1.61586707,  0.28407363, -0.02568776,
        1.49677566, -0.59023161,  0.41659155,  1.6137853 ,  0.08057172,
       -0.05392229,  1.01524393, -0.36986163,  0.52457967,  1.48737034,
       -0.66096022, -0.16360242,  0.54694754,  1.37075536])

## Ridge Regression

Regularization will usually work by adding a penalization term to the error. **Ridge Regression** adds a penalty term bassed on the squared value of the coefficients.

In [19]:
from sklearn.linear_model import Ridge

In [20]:
ridge_model = Ridge(alpha=10)

In [21]:
ridge_model.fit(X_train,y_train)

,alpha,10
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [22]:
test_predictions = ridge_model.predict(X_test)

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [24]:
MAE = mean_absolute_error(y_test,test_predictions)

In [25]:
MAE

0.5774404204714166

In [26]:
RMSE = np.sqrt(mean_squared_error(y_test,test_predictions))

In [27]:
RMSE

np.float64(0.8946386461319648)

Cross-validation checks the results with a variety of Alpha parameters (Lambda parameter in the math formula) to see  which one gets the better results.

In [28]:
from sklearn.linear_model import RidgeCV

In [29]:
ridge_cv_model = RidgeCV(alphas=(0.1,1.0,10.0))

In [30]:
ridge_cv_model.fit(X_train, y_train)

,alphas,"(0.1, ...)"
,fit_intercept,True
,scoring,None
,cv,None
,gcv_mode,None
,store_cv_results,False
,alpha_per_target,False


In [31]:
ridge_cv_model.alpha_

np.float64(0.1)

In [32]:
from sklearn.metrics import get_scorer_names

In [33]:
all_scores = get_scorer_names()

In [34]:
all_scores

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'd2_absolute_error_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_max_error',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'neg_root_mean_squared_log_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 're

In [35]:
ridge_cv_model = RidgeCV(alphas=(0.1,1.0,10.0),scoring='neg_mean_absolute_error')

In [36]:
ridge_cv_model.fit(X_train, y_train)

,alphas,"(0.1, ...)"
,fit_intercept,True
,scoring,'neg_mean_absolute_error'
,cv,None
,gcv_mode,None
,store_cv_results,False
,alpha_per_target,False


In [37]:
ridge_cv_model.alpha_

np.float64(0.1)

In [38]:
test_predictions = ridge_cv_model.predict(X_test)

In [39]:
MAE = mean_absolute_error(y_test,test_predictions)

In [40]:
MAE

0.42737748843375084

In [41]:
RMSE = np.sqrt(mean_squared_error(y_test,test_predictions))

In [42]:
RMSE

np.float64(0.6180719926926787)

In [43]:
ridge_cv_model.coef_

array([ 5.40769392,  0.5885865 ,  0.40390395, -6.18263924,  4.59607939,
       -1.18789654, -1.15200458,  0.57837796, -0.1261586 ,  2.5569777 ,
       -1.38900471,  0.86059434,  0.72219553, -0.26129256,  0.17870787,
        0.44353612, -0.21362436, -0.04622473, -0.06441449])

# LASSO - Least Absolute Shrinkage and Selection Operator

In [44]:
from sklearn.linear_model import LassoCV

In [45]:
lasso_cv_model = LassoCV(eps=0.1,alphas=100,cv=5)
# You can later adjust by asking the model to check alphas with smaller eps separation betweeen them
# or with more iterations adding the parameter max_iter=100000 (or more)

In [46]:
lasso_cv_model.fit(X_train,y_train)

,eps,0.1
,n_alphas,'deprecated'
,alphas,100
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,copy_X,True
,cv,5
,verbose,False
,n_jobs,None


In [47]:
lasso_cv_model.alpha_

np.float64(0.4943070909225831)

In [48]:
test_predictions = lasso_cv_model.predict(X_test)

In [49]:
MAE=mean_absolute_error(y_test,test_predictions)

In [50]:
MAE

0.6541723161252868

In [51]:
RMSE = np.sqrt(mean_squared_error(y_test,test_predictions))

In [52]:
RMSE



np.float64(1.1308001022762548)

In [53]:
lasso_cv_model.coef_

array([1.002651  , 0.        , 0.        , 0.        , 3.79745279,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        ])

# Elastic net: combining Lasso and Ridge

In [54]:
from sklearn.linear_model import ElasticNetCV

In [55]:
elastic_model = ElasticNetCV(l1_ratio=[0.1,0.5,0.7,0.9,0.95,0.99,1.0], eps=0.001,alphas=100,max_iter=1000000)

In [56]:
elastic_model.fit(X_train,y_train)

,l1_ratio,"[0.1, 0.5, ...]"
,eps,0.001
,n_alphas,'deprecated'
,alphas,100
,fit_intercept,True
,precompute,'auto'
,max_iter,1000000
,tol,0.0001
,cv,None
,copy_X,True
,verbose,0


In [59]:
#L1 ratios tried:
elastic_model.l1_ratio

[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

In [60]:
# Most successful L1 ratio
elastic_model.l1_ratio_

np.float64(1.0)

It means go for LASSO directly, it worked better.

In fact: let's compare the alphas used by LASSO before and this Elastic model:

In [64]:
lasso_cv_model = LassoCV(eps=0.001,alphas=100,cv=5, max_iter=1000000)

In [65]:
lasso_cv_model.fit(X_train,y_train)

,eps,0.001
,n_alphas,'deprecated'
,alphas,100
,fit_intercept,True
,precompute,'auto'
,max_iter,1000000
,tol,0.0001
,copy_X,True
,cv,5
,verbose,False
,n_jobs,None


In [66]:
lasso_cv_model.alpha_

np.float64(0.004943070909225831)

In [67]:
elastic_model.alpha_

np.float64(0.004943070909225831)

In [68]:
test_predictions = elastic_model.predict(X_test)

In [69]:
MAE = mean_absolute_error(y_test,test_predictions)

In [70]:
MAE

0.4335034618590074